# 下一小时预测与独立验证

训练程序负责三个扩展窗口与最终重训；本Notebook检查已实际执行的结果和无泄漏边界。

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT=Path.cwd()
assert (ROOT/'src').exists()
pd.set_option('display.max_columns',20)

In [2]:
protocol=json.loads((ROOT/'models/feature_protocol.json').read_text(encoding='utf-8'))
protocol

{'target': 'recorded pickup count in [t,t+1h), predicted at t',
 'features': ['zone_id',
  'hour',
  'weekday',
  'month',
  'is_weekend',
  'is_holiday',
  'lag_1',
  'lag_2',
  'lag_24',
  'lag_168',
  'roll_mean_3',
  'roll_mean_24',
  'roll_mean_168',
  'roll_std_24',
  'roll_std_168'],
 'excluded_features': ['target hour orders',
  'fares',
  'duration',
  'dropoff outcomes',
  'weather reanalysis',
  'future weather'],
 'time_zone': 'America/New_York',
 'dst': 'Ambiguous/nonexistent local hours are NaN. Targets and every lag/rolling/baseline window touching these NaNs are excluded, using one common evaluation universe.',
 'first_eligible_hour': '2025-01-29 00:00:00',
 'grid_rows': 2303880,
 'eligible_rows': 2036672,
 'assumption': 'Retrospective rolling one-hour evaluation assumes previous completed pickup-hour counts available at t; monthly publisher data cannot establish real-time arrival latency.'}

In [3]:
selection=json.loads((ROOT/'models/model_selection.json').read_text(encoding='utf-8'))
assert selection['test_used_for_selection'] is False
pd.DataFrame(selection['splits'])

,validation_month,train_n,validation_n,train_end,validation_start,validation_end
0,9,1311844,189360,2025-08-31 23:00:00,2025-09-01 00:00:00,2025-09-30 23:00:00
1,10,1501204,195672,2025-09-30 23:00:00,2025-10-01 00:00:00,2025-10-31 23:00:00
2,11,1696876,144124,2025-10-31 23:00:00,2025-11-01 00:00:00,2025-11-30 23:00:00


In [4]:
validation=pd.read_csv(ROOT/'reports/tables/validation_metrics.csv')
validation.pivot(index='model_label',columns='month',values='wape')

month,9,10,11
model_label,,,
LightGBM（31叶）,0.159935,0.149896,0.164418
LightGBM（63叶）,0.158483,0.147818,0.161287
前一周同小时,0.245539,0.204477,0.268495
前一天同小时,0.312402,0.295269,0.329514
四周同星期同小时均值,0.239390,0.170988,0.242745


In [5]:
test=pd.read_csv(ROOT/'reports/tables/test_metrics.csv')
print('验证选定:',selection['selected_label'])
test[['model_label','selected','n','wape','mae','rmse']]

验证选定: LightGBM（63叶）


,model_label,selected,n,wape,mae,rmse
0,前一天同小时,0,195672,0.293504,6.445991,21.339067
1,前一周同小时,0,195672,0.349627,7.678564,25.806107
2,四周同星期同小时均值,0,195672,0.299080,6.568442,21.568460
3,LightGBM（31叶）,0,195672,0.167294,3.674145,10.511072
4,LightGBM（63叶）,1,195672,0.164044,3.602758,10.318836


In [6]:
from src.features import zone_features
import numpy as np
idx=pd.date_range('2025-01-01',periods=1400,freq='h')
a=pd.Series(np.arange(1400)%13,index=idx)
b=a.copy();b.iloc[1000:]=999
pd.testing.assert_frame_equal(zone_features(a,1).loc[:1000,protocol['features']],zone_features(b,1).loc[:1000,protocol['features']])
print('改变未来数据不影响过去特征：通过')

改变未来数据不影响过去特征：通过


逐小时预测可以使用此前已经结束的测试小时，但不能利用目标小时的真实订单，也不能在12月上重新选模型。五种方案共享同一有效样本，零分母WAPE为空。